# Task 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import requests
from tqdm import tqdm
from pyspark.sql.functions import isnan, when, count, col

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
spark = SparkSession.builder \
    .appName("NYC Yellow Taxi Big Data") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory","8g") \
    .config("spark.executor.memory","8g") \
    .getOrCreate()

In [ ]:
df = spark.read.parquet(
    "/content/drive/MyDrive/Yellow Taxi Trip Records/*.parquet"
)

In [ ]:
df.show(20, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2026-03-01 00:02:26 |2026-03-01 00:13:45  |1              |2.58         |1         |N                 |48          |151 

In [ ]:
total_rows = df.count()

print("Total Rows:", total_rows)

Total Rows: 14908446


In [ ]:
print("Columns:", len(df.columns))

Columns: 20


In [ ]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [ ]:
print("Partitions :", df.rdd.getNumPartitions())

Partitions : 3


In [ ]:
folder = "/content/drive/MyDrive/Yellow Taxi Trip Records"

files = [
    "yellow_tripdata_2026-01.parquet",
    "yellow_tripdata_2026-02.parquet",
    "yellow_tripdata_2026-03.parquet",
    "yellow_tripdata_2026-04.parquet"
]

total_size = 0

print("Individual File Sizes")
print("-" * 60)

for file in files:
    path = os.path.join(folder, file)
    size_bytes = os.path.getsize(path)
    size_mb = size_bytes / (1024 ** 2)

    print(f"{file:<35} {size_mb:.2f} MB")

    total_size += size_bytes

print("-" * 60)
print(f"Total Dataset Size: {total_size / (1024 ** 2):.2f} MB")

Individual File Sizes
------------------------------------------------------------
yellow_tripdata_2026-01.parquet     61.19 MB
yellow_tripdata_2026-02.parquet     55.96 MB
yellow_tripdata_2026-03.parquet     64.75 MB
yellow_tripdata_2026-04.parquet     61.82 MB
------------------------------------------------------------
Total Dataset Size: 243.72 MB
